In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/Text Miners/Actual Work'

!pip install numpy --upgrade --quiet
!pip install scikit-learn pandas ftfy gensim --quiet
print('Done')

Mounted at /content/drive
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
Done


In [5]:
import pandas as pd
import pickle
import sys
import os

# Load VADER labelled reviews
df_sentiment = pd.read_csv(f'{BASE}/data/labelled_reviews.csv')
print('Sentiment labels shape:', df_sentiment.shape)
print(df_sentiment['label'].value_counts())

# Load LDA topic assignments
df_topics = pd.read_csv(f'{BASE}/task2-topics/lda_topic_assignments.csv')
print('\nTopic assignments shape:', df_topics.shape)
print(df_topics['lda_topic_label'].value_counts())

Sentiment labels shape: (380505, 4)
label
positive    359559
neutral      14784
negative      6162
Name: count, dtype: int64

Topic assignments shape: (380505, 8)
lda_topic_label
General Positive Sentiment    251765
Overall Experience             72933
Complaints & Issues            25196
Transport & Accessibility      10081
Check-in & Arrival              9079
Neighbourhood & Dining          7729
Facilities & Amenities          3722
Name: count, dtype: int64


In [6]:
# Keep only reviews where VADER is confident
# compound > 0.5  → clearly positive
# compound < -0.5 → clearly negative
# -0.2 to 0.2    → clearly neutral
high_conf = df_sentiment[
    (df_sentiment['compound_score'] > 0.5) |
    (df_sentiment['compound_score'] < -0.5) |
    ((df_sentiment['compound_score'] >= -0.2) &
     (df_sentiment['compound_score'] <= 0.2))
]

print(f'Original reviews:        {len(df_sentiment):,}')
print(f'High confidence reviews: {len(high_conf):,}')
print(f'Filtered out:            {len(df_sentiment) - len(high_conf):,}')
print(f'\nLabel distribution after filtering:')
print(high_conf['label'].value_counts())

df_sentiment = high_conf[['listing_id', 'comments', 'label', 'compound_score']]

Original reviews:        380,505
High confidence reviews: 351,467
Filtered out:            29,038

Label distribution after filtering:
label
positive    337050
neutral      10099
negative      4318
Name: count, dtype: int64


In [7]:
# Merge VADER sentiment with LDA topic assignments
df_merged = df_sentiment.merge(
    df_topics[['listing_id', 'comments', 'lda_topic_label', 'lda_topic_id']],
    on=['listing_id', 'comments'],
    how='inner'
)

df_merged = df_merged.rename(columns={
    'label': 'sentiment',
    'lda_topic_label': 'aspect'
})

print('Merged shape:', df_merged.shape)
print('\nAspect distribution:')
print(df_merged['aspect'].value_counts())
print('\nSentiment distribution:')
print(df_merged['sentiment'].value_counts())

Merged shape: (351467, 6)

Aspect distribution:
aspect
General Positive Sentiment    231435
Overall Experience             70427
Complaints & Issues            21621
Transport & Accessibility       9055
Check-in & Arrival              8126
Neighbourhood & Dining          7264
Facilities & Amenities          3539
Name: count, dtype: int64

Sentiment distribution:
sentiment
positive    337050
neutral      10099
negative      4318
Name: count, dtype: int64


In [8]:
# Sample 500 reviews ensuring coverage across all 7 aspects
# and balanced sentiment within each aspect
samples_per_aspect = 500 // 7  # ~42 per aspect

sampled = []
for aspect in df_merged['aspect'].unique():
    aspect_df = df_merged[df_merged['aspect'] == aspect]
    for sentiment in ['positive', 'neutral', 'negative']:
        sent_df = aspect_df[aspect_df['sentiment'] == sentiment]
        n = min(len(sent_df), max(1, samples_per_aspect // 3))
        if len(sent_df) > 0:
            sampled.append(sent_df.sample(n, random_state=42))

training_df = pd.concat(sampled).drop_duplicates().reset_index(drop=True)
print(f'Training samples: {len(training_df)}')
print('\nAspect distribution:')
print(training_df['aspect'].value_counts())
print('\nSentiment distribution:')
print(training_df['sentiment'].value_counts())

Training samples: 477

Aspect distribution:
aspect
General Positive Sentiment    69
Overall Experience            69
Complaints & Issues           69
Check-in & Arrival            69
Transport & Accessibility     69
Facilities & Amenities        69
Neighbourhood & Dining        63
Name: count, dtype: int64

Sentiment distribution:
sentiment
positive    161
neutral     161
negative    155
Name: count, dtype: int64


In [9]:
spot_check = training_df.sample(50, random_state=99)
print('SPOT CHECK — verify these look correct:\n')
for _, row in spot_check.iterrows():
    print(f'  Aspect:    {row["aspect"]}')
    print(f'  Sentiment: {row["sentiment"]}')
    print(f'  Comment:   {row["comments"][:120]}')
    print(f'  {"-"*60}')

SPOT CHECK — verify these look correct:

  Aspect:    General Positive Sentiment
  Sentiment: neutral
  Comment:   A very decent place to stay in bangkok
  ------------------------------------------------------------
  Aspect:    Complaints & Issues
  Sentiment: negative
  Comment:   I was really disappointed with the quality of the accommodation. At first the air conditioning had a smell which forced 
  ------------------------------------------------------------
  Aspect:    Complaints & Issues
  Sentiment: neutral
  Comment:   The apartment is nice and had everything I need for my stay. Internet is shared between two apartments, fast and reliabl
  ------------------------------------------------------------
  Aspect:    Facilities & Amenities
  Sentiment: positive
  Comment:   cld be better equipped such as better kitchen utensils. wine opener. cutleries. Some lights were not working.
  ------------------------------------------------------------
  Aspect:    Complaints & Issues
  S

In [10]:
sys.path.append(f'{BASE}/pre-processing')
import preprocess

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# Load and preprocess FULL dataset for TF-IDF vocabulary
print('Preprocessing full dataset (this takes a few minutes)...')
df_full = pd.read_csv(f'{BASE}/data/MergedCleaned.csv')
df_full = preprocess.preprocess_dataframe(df_full)
print(f'Full dataset preprocessed: {len(df_full):,} rows')

# Fit TF-IDF on FULL dataset — captures complete vocabulary
tfidf = TfidfVectorizer(max_features=10000)
tfidf.fit(df_full['cleaned_text'])
print(f'TF-IDF vocabulary: {len(tfidf.vocabulary_):,} words')

# Preprocess training samples and transform
print('\nPreprocessing training samples...')
training_df = preprocess.preprocess_dataframe(training_df)
X_train_features = tfidf.transform(training_df['cleaned_text'])

# Encode labels
le_aspect = LabelEncoder()
y_aspect = le_aspect.fit_transform(training_df['aspect'])

le_sentiment = LabelEncoder()
y_sentiment = le_sentiment.fit_transform(training_df['sentiment'])

print(f'Training feature matrix: {X_train_features.shape}')
print('Aspect classes:', le_aspect.classes_)
print('Sentiment classes:', le_sentiment.classes_)

Preprocessing full dataset (this takes a few minutes)...
Full dataset preprocessed: 380,505 rows
TF-IDF vocabulary: 10,000 words

Preprocessing training samples...
Training feature matrix: (477, 10000)
Aspect classes: ['Check-in & Arrival' 'Complaints & Issues' 'Facilities & Amenities'
 'General Positive Sentiment' 'Neighbourhood & Dining'
 'Overall Experience' 'Transport & Accessibility']
Sentiment classes: ['negative' 'neutral' 'positive']


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_tr, X_te, y_tr, y_te = train_test_split(
    X_train_features, y_aspect,
    test_size=0.2, random_state=42, stratify=y_aspect
)

lr_aspect = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=1.0
)
lr_aspect.fit(X_tr, y_tr)
y_pred_aspect = lr_aspect.predict(X_te)

print('=== Aspect Prediction (LR) ===')
print(classification_report(y_te, y_pred_aspect,
      target_names=le_aspect.classes_))

=== Aspect Prediction (LR) ===
                            precision    recall  f1-score   support

        Check-in & Arrival       0.89      0.57      0.70        14
       Complaints & Issues       0.67      0.86      0.75        14
    Facilities & Amenities       0.75      0.64      0.69        14
General Positive Sentiment       0.75      0.43      0.55        14
    Neighbourhood & Dining       0.71      0.83      0.77        12
        Overall Experience       0.59      0.71      0.65        14
 Transport & Accessibility       0.56      0.71      0.62        14

                  accuracy                           0.68        96
                 macro avg       0.70      0.68      0.67        96
              weighted avg       0.70      0.68      0.67        96



In [12]:
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_train_features, y_sentiment,
    test_size=0.2, random_state=42, stratify=y_sentiment
)

lr_sentiment = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    C=1.0
)
lr_sentiment.fit(X_tr_s, y_tr_s)
y_pred_sentiment = lr_sentiment.predict(X_te_s)

print('=== Sentiment Prediction (LR) ===')
print(classification_report(y_te_s, y_pred_sentiment,
      target_names=le_sentiment.classes_))

=== Sentiment Prediction (LR) ===
              precision    recall  f1-score   support

    negative       0.65      0.55      0.60        31
     neutral       0.55      0.70      0.61        33
    positive       0.68      0.59      0.63        32

    accuracy                           0.61        96
   macro avg       0.63      0.61      0.61        96
weighted avg       0.63      0.61      0.61        96



In [13]:
print('Running predictions on full dataset (this may take a few minutes)...')

# Transform full dataset using the fitted TF-IDF
X_full = tfidf.transform(df_full['cleaned_text'])

# Predict aspect and sentiment for every review
df_full['predicted_aspect']    = le_aspect.inverse_transform(
    lr_aspect.predict(X_full)
)
df_full['predicted_sentiment'] = le_sentiment.inverse_transform(
    lr_sentiment.predict(X_full)
)

print(f'Predictions done for {len(df_full):,} reviews')
print('\nPredicted aspect distribution:')
print(df_full['predicted_aspect'].value_counts())
print('\nPredicted sentiment distribution:')
print(df_full['predicted_sentiment'].value_counts())

Running predictions on full dataset (this may take a few minutes)...
Predictions done for 380,505 reviews

Predicted aspect distribution:
predicted_aspect
General Positive Sentiment    145743
Overall Experience             78983
Transport & Accessibility      40863
Neighbourhood & Dining         39192
Complaints & Issues            30257
Check-in & Arrival             23734
Facilities & Amenities         21733
Name: count, dtype: int64

Predicted sentiment distribution:
predicted_sentiment
positive    278729
neutral      75408
negative     26368
Name: count, dtype: int64


In [19]:
import os

# List all folders to find the right one
print("Folders in Actual Work:")
print(os.listdir(BASE))

# List files in the task3 folder (try both variations if needed)
target_folder = f'{BASE}/task3_combined'
if os.path.exists(target_folder):
    print(f"\nFiles in {target_folder}:")
    print(os.listdir(target_folder))
else:
    print(f"\nFolder {target_folder} does not exist yet!")

print(os.listdir(f"{BASE}/task3-combination"))

Folders in Actual Work:
['task3-combination', 'data', 'pre-processing', 'features', 'task1-sentiments', 'task2-topics', 'dashboard', 'BeforeYouStart.docx']

Folder /content/drive/MyDrive/Text Miners/Actual Work/task3_combined does not exist yet!
['task3_supervised.ipynb', 'nss_scores.csv', 'task3_rulebased.ipynb', 'task3_supervised_output.csv']


In [21]:
# Change 'task3-combination' to 'task3_combined'
nss_df = pd.read_csv(f'{BASE}/task3-combination/nss_scores.csv')

print('=== Rule-based NSS Summary ===')
print(nss_df.groupby('aspect')['NSS'].describe().round(3))

print('\n=== Approach Comparison ===')
print('''
Approach            | Strengths                           | Weaknesses
--------------------|-------------------------------------|--------------------------
Rule-based NSS      | Interpretable, no training needed   | Relies on keyword dict
                    | Direct score per aspect             | Misses context
--------------------|-------------------------------------|--------------------------
Supervised LR       | Learns from data, handles unseen    | Small training set
                    | phrasing, works on full dataset     | Some label noise
''')

=== Rule-based NSS Summary ===
                 count   mean    std  min    25%  50%  75%  max
aspect                                                         
amenities       9353.0  0.637  0.595 -1.0  0.429  1.0  1.0  1.0
check_in        5739.0  0.613  0.677 -1.0  0.359  1.0  1.0  1.0
cleanliness    13726.0  0.852  0.368 -1.0  0.900  1.0  1.0  1.0
communication  14454.0  0.894  0.314 -1.0  0.971  1.0  1.0  1.0
location       14290.0  0.885  0.290 -1.0  0.889  1.0  1.0  1.0
value           9954.0  0.878  0.366 -1.0  1.000  1.0  1.0  1.0

=== Approach Comparison ===

Approach            | Strengths                           | Weaknesses
--------------------|-------------------------------------|--------------------------
Rule-based NSS      | Interpretable, no training needed   | Relies on keyword dict
                    | Direct score per aspect             | Misses context
--------------------|-------------------------------------|--------------------------
Supervised LR       | Lear

In [22]:
# Save full predictions
output_path = f'{BASE}/task3-combination/task3_supervised_output.csv'
df_full[[
    'listing_id', 'neighbourhood', 'room_type', 'comments',
    'cleaned_text', 'predicted_aspect', 'predicted_sentiment'
]].to_csv(output_path, index=False)

print(f'✅ Exported: task3_supervised_output.csv')
print(f'   {len(df_full):,} rows')
print('\nMessage the group — task3_supervised_output.csv is ready in /task3-combination/')

✅ Exported: task3_supervised_output.csv
   380,505 rows

Message the group — task3_supervised_output.csv is ready in /task3-combination/
